# Methods demos: thresholds, composites, correlation and chance

**Optional companion to `student_project.ipynb`.** It needs only `oni.csv` (no ERA5 files), so it runs in seconds.

Each section matches a methods block in the writing-strand tutorials:

| Section | Idea | Tutorial |
|---|---|---|
| 1 | Thresholding the ONI: how El Niño / La Niña months are chosen | T3 (15 Oct) |
| 2 | Building a composite, step by step | T3 |
| 3 | Coins in a bucket: why *r* alone is not enough | T5 (12 Nov) |
| 4 | Calculate a p-value yourself | T5 |
| 5 | Effective sample size: why 247 months are not 247 independent samples | T5 |
| 6 | A correlation map made from pure noise | T5 |
| 7 | (Paste into the main notebook) Is my region's correlation real? A shift test | T5–T6 |

Anything labelled **synthetic** is made-up data used to show a method. It is **not** ERA5, and it doesn't go in your report.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import factorial
from scipy import stats
%matplotlib inline
rng = np.random.default_rng(2026)

raw = pd.read_csv("oni.csv", skiprows=1, header=None, names=["date", "oni"])
raw["date"] = pd.to_datetime(raw["date"]); oni_all = raw.set_index("date")["oni"]
oni_all[oni_all <= -99] = np.nan
oni = oni_all["2005-01":"2025-08"].dropna()      # same period as the main notebook
x = oni.values; n = len(x)
print(f"ONI months in the analysis period: {n}  ({oni.index[0]:%b %Y} to {oni.index[-1]:%b %Y})")

## 1. Thresholding the ONI

The main notebook classes **each month** as El Niño if ONI ≥ `ENSO_THRESH` and La Niña if ONI ≤ −`ENSO_THRESH` (default 0.5 °C). NOAA's *official* episode definition is stricter: the ±0.5 °C threshold must be met for **at least five consecutive overlapping 3-month seasons**. Your Methods should say which you used.

**Try it:** change `THRESH` and re-run. How many months are left in each group at 1.0 °C? Would you trust a composite built from them?

In [ ]:
THRESH = 0.5     # try 1.0 and 1.5

state = np.where(x >= THRESH, "El Niño", np.where(x <= -THRESH, "La Niña", "Neutral"))
print(pd.Series(state).value_counts().reindex(["El Niño", "Neutral", "La Niña"]))

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.fill_between(oni.index, THRESH, x, where=x >= THRESH, color="tab:red", interpolate=True)
ax.fill_between(oni.index, -THRESH, x, where=x <= -THRESH, color="tab:blue", interpolate=True)
ax.plot(oni.index, x, "k", lw=1)
for v in (THRESH, -THRESH): ax.axhline(v, ls="--", c="grey")
ax.set_ylabel("ONI (°C)"); ax.set_title(f"Months beyond ±{THRESH} °C"); plt.show()

# Which months does one big event contribute?
print("El Niño months by calendar year:")
print(pd.Series(oni.index.year[x >= THRESH]).value_counts().sort_index().to_string())

## 2. Building a composite, step by step

A composite answers the question: **on average, how different is the anomaly in El Niño months from La Niña months?**

1. Remove the seasonal cycle to get **anomalies** (done for you in the main notebook).
2. Select the El Niño months and **average** their anomalies. Do the same for La Niña (and neutral).
3. **Subtract:** El Niño − La Niña.

Below, the ONI is real and the cloud anomaly is **synthetic**. Notice the asymmetry: this made-up cloud responds strongly to El Niño and hardly at all to La Niña. The single difference value hides that, and the phase-by-phase means show it.

In [ ]:
# synthetic cloud anomaly that responds mainly to El Niño (illustrative only)
cloud = np.where(x > 0.5, -2.5 * (x - 0.5), 0.0) + 0.8 * rng.standard_normal(n)

st = pd.Series(np.where(x >= 0.5, "El Niño", np.where(x <= -0.5, "La Niña", "Neutral")))
means = pd.Series(cloud).groupby(st).agg(["mean", "count"]).reindex(["El Niño", "Neutral", "La Niña"])
print(means.round(2))
print(f"\nComposite difference, El Niño − La Niña: {means.loc['El Niño','mean'] - means.loc['La Niña','mean']:+.2f} percentage points")
print("Ask: is the difference driven by El Niño, La Niña, or both? (The main notebook prints the same table for real data in sections 3 and 4.)")

## 3. Coins in a bucket: why *r* alone is not enough

Put the coins 1p, 2p, 5p, 10p, 20p, 50p and £1 in a bucket and pull them out at random. What is the chance they come out **in perfect order**, smallest to largest? Then the correlation between draw order and coin value is a perfect *r* = 1, and it happened by pure chance.

There are *n*! possible orders and only one is perfectly ascending:

In [ ]:
coins = [1, 2, 5, 10, 20, 50, 100]
for k in range(2, 8):
    print(f"{k} coins: chance of perfect order = 1/{factorial(k):>4} = {1/factorial(k):.4f}")

# Simulate it: draw the coins at random 200,000 times and look at the correlation
def draw_r(k, trials=200_000):
    vals = np.array(coins[:k], float)
    draws = np.array([rng.permutation(vals) for _ in range(trials)])
    order = np.arange(k, dtype=float)
    d = draws - draws.mean(1, keepdims=True); o = order - order.mean()
    return (d * o).sum(1) / np.sqrt((d**2).sum(1) * (o**2).sum())

fig, ax = plt.subplots(figsize=(8, 4))
for k, c in [(3, "tab:orange"), (7, "tab:blue")]:
    r = draw_r(k)
    ax.hist(r, bins=50, range=(-1, 1), density=True, histtype="step", lw=2, color=c,
            label=f"{k} coins: |r| ≥ 0.8 in {(np.abs(r) >= 0.8).mean()*100:.1f}% of draws")
ax.set_xlabel("Correlation between draw order and coin value"); ax.set_ylabel("Density")
ax.legend(); ax.set_title("Same random process; a big r is common with few coins"); plt.show()

**The point:** *r* = 0.9 from 5 points is unremarkable; *r* = 0.3 from 247 independent points would be very unlikely by chance. You need *r* **and** *n* to judge a correlation. The p-value combines them.

## 4. Calculate a p-value yourself

For a Pearson correlation *r* from *n* independent pairs, the test statistic is

$$t = \frac{r\sqrt{n-2}}{\sqrt{1-r^2}}$$

with *n* − 2 degrees of freedom. The two-tailed p-value is the chance of a |*t*| at least this large if the true correlation were zero.

**Your task (workbook Task 5D):**
1. Take *r* and *n* for your region from the panel in section 5 of the main notebook.
2. Put them in below and get *p*.
3. Now replace *n* with **n_eff** from the section 6 table. What happens to *p*? Which would you report, and why?

In [ ]:
def p_from_r(r, n):
    t = r * np.sqrt(n - 2) / np.sqrt(1 - r**2)
    return t, 2 * (1 - stats.t.cdf(abs(t), df=n - 2))

MY_R, MY_N, MY_NEFF = -0.45, 247, 60      # <-- replace with your own numbers

for label, nn in [("using n", MY_N), ("using n_eff", MY_NEFF)]:
    t, p = p_from_r(MY_R, nn)
    print(f"{label:12s}: n = {nn:5.0f}   t = {t:6.2f}   p = {p:.2g}")

print("\nSmallest |r| that is 'significant' at p < 0.05 for different n:")
for nn in [5, 10, 30, 60, 100, 247]:
    tc = stats.t.ppf(0.975, nn - 2); print(f"  n = {nn:3d}:  |r| > {tc/np.sqrt(nn-2+tc**2):.2f}")

## 5. Effective sample size

Monthly ENSO data have long memory: this month's ONI is very like last month's. So 247 months contain far fewer than 247 **independent** pieces of information. The main notebook uses a Bretherton-style correction:

$$n_\text{eff} = n\,\frac{1 - r_{1x}\,r_{1y}}{1 + r_{1x}\,r_{1y}}$$

where *r*₁ is the lag-1 autocorrelation of each series.

In [ ]:
def r1(z):
    z = z - z.mean(); return np.corrcoef(z[:-1], z[1:])[0, 1]
r1x = r1(x)
print(f"Lag-1 autocorrelation of the ONI: {r1x:.2f}")
for r1y in (0.0, 0.3, 0.5, 0.8):
    print(f"  cloud anomaly with r1 = {r1y}:  n_eff = {n*(1-r1x*r1y)/(1+r1x*r1y):5.0f}   (from n = {n})")

## 6. A correlation map made from pure noise

Below, the **real** ONI is correlated with a field of **pure random noise** that has realistic memory in time and smooth blobs in space, like real weather. Contours mark grid points that would pass a naive 5% significance test (using *n*, not *n*_eff).

**Ask yourself:** how much of this map would you have believed? And remember that even with no autocorrelation at all, about 5% of grid points pass by chance. On a map with thousands of points, that is hundreds of spurious "signals".

In [ ]:
from scipy.ndimage import gaussian_filter
lat = np.arange(-60, 61, 2.5); lon = np.arange(-180, 180, 2.5)
noise = np.empty((n, lat.size, lon.size)); e = rng.standard_normal(noise.shape)
noise[0] = e[0]
for t in range(1, n): noise[t] = 0.6 * noise[t-1] + 0.8 * e[t]          # memory in time
for t in range(n): noise[t] = gaussian_filter(noise[t], (2.5, 4), mode=("nearest", "wrap"))   # blobs in space
xa = (x - x.mean()) / x.std(); na = (noise - noise.mean(0)) / noise.std(0)
rmap = (xa[:, None, None] * na).mean(0)
rcrit = 1.96 / np.sqrt(n)
fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.pcolormesh(lon, lat, rmap, cmap="RdBu_r", vmin=-0.5, vmax=0.5, shading="auto")
ax.contour(lon, lat, np.abs(rmap) > rcrit, levels=[0.5], colors="k", linewidths=0.7)
plt.colorbar(im, ax=ax, label="r with ONI"); ax.set_aspect("equal")
ax.set_title(f"Real ONI vs pure noise: {(np.abs(rmap) > rcrit).mean()*100:.0f}% of points pass a naive 5% test"); plt.show()

## 7. Is my region's correlation real? A shift test (paste into the main notebook)

A robust check that needs no formula. Slide the ONI series along in time by every possible amount of 12 months or more, wrapping round at the end, and recompute *r* each time. Shifting keeps both series' memory intact but destroys any real alignment, so the shifted correlations show what chance alone produces **for data like yours**.

Copy the cell below to the **end of `student_project.ipynb`** (after section 6 has run) and set `MY_REGION`.

In [ ]:
# ---- paste into student_project.ipynb after section 6 ----
MY_REGION = "se_pacific_sc"     # any key from REGIONS, e.g. "nino4", "maritime_continent"

try:
    xy = pd.concat([oni_r, regional_anom[MY_REGION]], axis=1).dropna()
    xo, yo = xy.iloc[:, 0].values, xy.iloc[:, 1].values
    r_obs = np.corrcoef(xo, yo)[0, 1]
    shifts = range(12, len(xo) - 11)
    r_null = np.array([np.corrcoef(np.roll(xo, s), yo)[0, 1] for s in shifts])
    p_shift = (np.abs(r_null) >= abs(r_obs)).mean()
    plt.hist(r_null, bins=30, color="grey"); plt.axvline(r_obs, color="tab:red", lw=2)
    plt.xlabel("r from shifted ONI (chance)"); plt.title(f"{REGIONS[MY_REGION]['label']}: observed r = {r_obs:.2f}")
    plt.show()
    print(f"Observed r = {r_obs:.2f}; shifted correlations as extreme: {p_shift*100:.1f}% of {len(r_null)} shifts")
except NameError:
    print("This cell needs `oni_r` and `regional_anom` from the main notebook. Paste it there after section 6.")